# 🚀 Novel Visualiser — ComfyUI Cloud Runner (Kaggle & Colab T4 GPU)

This notebook runs **ComfyUI** on a free Kaggle / Colab **T4 GPU** with all required models installed:
- **Checkpoint**: `Wai_Illustrious_v16.safetensors` (SDXL anime/illustration master model)
- **IP-Adapter**: `ip-adapter-plus_sdxl_vit-h.safetensors` (Character biometric consistency)
- **CLIP-Vision**: `CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors` (Vision feature encoder)
- **VAE**: `sdxl.vae.safetensors` (SDXL FP16 VAE fix)
- **Custom Nodes**: `ComfyUI_IPAdapter_plus` & `ComfyUI-Florence2`
- **Cloudflare Tunnel**: Free public tunnel with **no account required**.

---
### ⚡ Quick Start:
1. **Kaggle**: Turn on **GPU (T4 x2 or P100)** under *Notebook settings* -> *Accelerator*, and enable **Internet** (Settings -> Internet ON).
2. **Google Colab**: Select *Runtime* -> *Change runtime type* -> **T4 GPU**.
3. Run Cells 1, 2, 3, 4 sequentially.
4. Copy the printed Cloudflare URL (e.g. `https://xxxx.trycloudflare.com`) into your local `novel_visualiser/.env`:
   ```env
   COMFYUI_URL=https://xxxx.trycloudflare.com
   ```

In [ ]:
# @title 1. Install ComfyUI, aria2 (fast downloader), & PyTorch
import os

# Determine platform working directory
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
os.chdir(WORK_DIR)
print(f"Working in: {WORK_DIR}")

# Install aria2 for multi-threaded high-speed model downloads
!apt-get update -qq && apt-get install -y -qq aria2

# Clone ComfyUI if not already present
if not os.path.exists(f"{WORK_DIR}/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd {WORK_DIR}/ComfyUI
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q httpx pyngrok timm einops transformers
print("✅ ComfyUI core dependencies installed successfully!")

In [ ]:
# @title 2. Install Custom Nodes (IPAdapter Plus & Florence-2)
import os
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
nodes_dir = f"{WORK_DIR}/ComfyUI/custom_nodes"
os.makedirs(nodes_dir, exist_ok=True)
%cd {nodes_dir}

# 1. ComfyUI IPAdapter Plus
if not os.path.exists("ComfyUI_IPAdapter_plus"):
    !git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git

# 2. ComfyUI Florence-2 (Auto-captioning & visual tagging)
if not os.path.exists("ComfyUI-Florence2"):
    !git clone https://github.com/kijai/ComfyUI-Florence2.git

%cd ComfyUI-Florence2
!pip install -q -r requirements.txt

%cd {WORK_DIR}/ComfyUI
print("✅ Custom nodes (IPAdapter Plus & Florence-2) installed!")

In [ ]:
# @title 3. Download Models with aria2 (Fast Parallel Downloads ~60-90s)
import os
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
base_dir = f"{WORK_DIR}/ComfyUI"

ckpt_dir = f"{base_dir}/models/checkpoints"
ip_dir = f"{base_dir}/models/ipadapter"
clip_dir = f"{base_dir}/models/clip_vision"
vae_dir = f"{base_dir}/models/vae"

for d in [ckpt_dir, ip_dir, clip_dir, vae_dir]:
    os.makedirs(d, exist_ok=True)

print("Downloading models... (this takes ~1-2 minutes on Kaggle's 100MB/s network)")

# 1. Wai Illustrious v1.6 Checkpoint (6.9 GB) — Saved as Wai_Illustrious_v16.safetensors
target_ckpt = f"{ckpt_dir}/Wai_Illustrious_v16.safetensors"
if not os.path.exists(target_ckpt):
    print("\n📥 Downloading Wai_Illustrious_v16.safetensors...")
    !aria2c -c -x 16 -s 16 -k 1M "https://huggingface.co/seraphimzz/checkpoints/resolve/main/waiIllustriousSDXL_v160.safetensors" -d "{ckpt_dir}" -o "Wai_Illustrious_v16.safetensors"
else:
    print("\n⚡ Wai_Illustrious_v16.safetensors already exists!")

# 2. IP-Adapter Plus SDXL ViT-H (808 MB)
target_ipa = f"{ip_dir}/ip-adapter-plus_sdxl_vit-h.safetensors"
if not os.path.exists(target_ipa):
    print("\n📥 Downloading ip-adapter-plus_sdxl_vit-h.safetensors...")
    !aria2c -c -x 8 -s 8 -k 1M "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors" -d "{ip_dir}" -o "ip-adapter-plus_sdxl_vit-h.safetensors"
else:
    print("\n⚡ ip-adapter-plus_sdxl_vit-h.safetensors already exists!")

# 3. CLIP-Vision ViT-H (2.4 GB)
target_clip = f"{clip_dir}/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
if not os.path.exists(target_clip):
    print("\n📥 Downloading CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors...")
    !aria2c -c -x 16 -s 16 -k 1M "https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors" -d "{clip_dir}" -o "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
else:
    print("\n⚡ CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors already exists!")

# 4. SDXL VAE Fix (319 MB)
target_vae = f"{vae_dir}/sdxl.vae.safetensors"
if not os.path.exists(target_vae):
    print("\n📥 Downloading sdxl.vae.safetensors...")
    !aria2c -c -x 4 -s 4 -k 1M "https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors" -d "{vae_dir}" -o "sdxl.vae.safetensors"

print("\n=======================================================")
print("   MODEL DOWNLOAD VERIFICATION:")
print("=======================================================")
for path in [target_ckpt, target_ipa, target_clip, target_vae]:
    if os.path.exists(path):
        sz_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  [FOUND] {os.path.basename(path)} ({sz_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {os.path.basename(path)}")
print("=======================================================")

In [ ]:
# @title 4. Launch ComfyUI with Free Cloudflare Tunnel
import subprocess
import threading
import time
import re
import os

WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
%cd {WORK_DIR}/ComfyUI

# Download Cloudflared tunnel binary
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Start ComfyUI in background with T4 GPU optimizations
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--preview-method", "auto",
    "--lowvram"
])

# Wait for ComfyUI server to bind socket
time.sleep(5)

# Start Cloudflare tunnel
cf_proc = subprocess.Popen([
    "/tmp/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188",
    "--metrics", "127.0.0.1:4444"
], stderr=subprocess.PIPE, text=True)

print("\n" + "="*65)
print("   CONNECTING COMFYUI TO CLOUDFLARE TUNNEL...")
print("="*65)

# Extract and display the public tunnel URL
tunnel_url = None
for line in cf_proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print(f"\n🎉 COMFYUI IS LIVE ON T4 GPU!")
        print(f"\nPublic Tunnel URL: {tunnel_url}")
        print(f"\n👉 Update your local novel_visualiser/.env with this line:")
        print(f"COMFYUI_URL={tunnel_url}\n")
        print("="*65)
        break

# Keep process alive
try:
    comfy_proc.wait()
except KeyboardInterrupt:
    comfy_proc.kill()
    cf_proc.kill()